In [5]:
import ee
import geemap

In [28]:
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project = "ee-my-usenvdi" )

In [29]:
delegacion_name = 'Iztapalapa' 

# 1. Get the boundary feature
mexico_city = ee.FeatureCollection("FAO/GAUL/2015/level2") \
    .filter(ee.Filter.eq('ADM1_NAME', 'Distrito Federal')) \
    .filter(ee.Filter.stringContains('ADM2_NAME', delegacion_name))

# 2. CRITICAL FIX: Extract the actual geometry and calculate its exact center point
delegacion_geom = mexico_city.geometry()
center_point = delegacion_geom.centroid()

# Print the coordinates to confirm it's pointing to CDMX
coords = center_point.coordinates().getInfo()
print(f"Searching using center point coordinates: Longitude {coords[0]:.4f}, Latitude {coords[1]:.4f}")

# 3. Filter using ONLY the center point instead of the comple`x polygon
ecostress_all = ee.ImageCollection("NASA/ECOSTRESS/L2T_LSTE/V2") \
    .filterBounds(center_point)

total_images = ecostress_all.size().getInfo()
print(f"Total ECOSTRESS images EVER recorded over this point: {total_images}")

if total_images == 0:
    print("❌ Still 0 images. There may be a temporary Earth Engine catalog outage for this dataset.")
else:
    # Get the 20 most recent captures
    recent_images = ecostress_all.sort('system:time_start', False).limit(20)
    
    def get_date(img):
        return ee.Feature(None, {'date': ee.Date(img.get('system:time_start')).format('YYYY-MM-dd')})
    
    dates_info = recent_images.map(get_date).getInfo()
    dates_list = [feat['properties']['date'] for feat in dates_info['features']]
    print(f"✅ Found data! Most recent passes: {dates_list[:5]}")
    
    ecostress_recent = ee.ImageCollection(recent_images)

    def process_ecostress(image):
        cloud_mask = image.select('cloud').eq(0)
        lst_celsius = image.select('LST').subtract(273.15).rename('LST_Celsius')
        lst_clean = lst_celsius.updateMask(cloud_mask)
        return lst_clean.copyProperties(image, ['system:time_start'])

    ecostress_clean = ecostress_recent.map(process_ecostress)
    
    # 4. Clip the final result back to the complex geometry
    uhi_map = ecostress_clean.median().clip(delegacion_geom).unmask(-9999)

    vis_params = {
        'min': 25, 
        'max': 45, 
        'palette': [
            '040274', '040281', '0502a3', '0502b8', '0502ce', '0502e6',
            '0602ff', '235cb1', '307ef3', '269db1', '30c8e2', '32d3ef',
            '3be285', '3ff38f', '86e26f', '3ae237', 'b5e22e', 'd6e21f',
            'fff705', 'ffd611', 'ffb613', 'ff8b13', 'ff6e08', 'ff500d',
            'ff0000', 'de0101', 'c21301', 'a71001', '911003'
        ]
    }
    
    Map = geemap.Map()
    Map.centerObject(delegacion_geom, 13) 
    Map.addLayer(uhi_map.updateMask(uhi_map.neq(-9999)), vis_params, f'ECOSTRESS Recent LST - {delegacion_name}')
    Map.addLayer(ee.Image().paint(mexico_city, 0, 2), {'palette': 'black'}, 'Boundary')
    Map.add_colorbar(vis_params, label="Land Surface Temp (°C)", orientation="horizontal", layer_name="ECOSTRESS LST")

    # Export
    out_dir = os.getcwd()
    out_file = os.path.join(out_dir, f'uhi_clearsky_{delegacion_name}.tif')
    
    print("Starting export to local drive...")
    geemap.ee_export_image(
        uhi_map, 
        filename=out_file, 
        scale=70, 
        region=delegacion_geom, 
        file_per_band=False,
        crs='EPSG:4326'
    )
    print(f"✅ Download complete! Image saved to: {out_file}")

Map

Searching using center point coordinates: Longitude -99.0246, Latitude 19.3163
Total ECOSTRESS images EVER recorded over this point: 0
❌ Still 0 images. There may be a temporary Earth Engine catalog outage for this dataset.


Map(bottom=14945445.0, center=[19.281402951861907, -99.10439908504488], controls=(WidgetControl(options=['posi…